In [ ]:
# Load data, extract exchange rates, and normalize them
import pandas as pd
import numpy as np

# Load the CSV file into a pandas DataFrame
file_path = 'priceIndex_and_exchangeRate_series_2025_SP.csv'  # Replace with your CSV file path
data = pd.read_csv(file_path, index_col=0)

african_countries = [
    'DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CMR', 'CPV', 'CAF', 'TCD', 'COM', 'COG', 'CIV', 
    'COD', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN', 
    'LBY', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA', 
    'RWA', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA', 
    'ZMB', 'ZWE'
]

def build_hybrid_index_df(
    start="1990-01-01",
    end="2023-01-01",
    countries=african_countries,
    df=data
):
    """
    Create tidy dataframe with columns:
      country | year | hybrid_index
    where hybrid_index = (NER / local_price_index), normalized to 100 at first valid obs.
    """

    start_date = pd.to_datetime(start)
    df.index = pd.to_datetime(df.index)
    df = df[(df.index >= start_date) & (df.index <= end)]
    
    records = []

    for country in countries:
        panusatls_col = f"{country}PANUSATLS"    # nominal exchange rate (LCU per USD)
        price_level_col = f"{country}NECONPRVTXN"  # local CPI or GDP deflator

        if panusatls_col not in df.columns or price_level_col not in df.columns:
            print(f"Missing columns for {country}")
            continue

        series =   df[price_level_col] / df[panusatls_col]
        first_valid = series.first_valid_index()
        if first_valid is None:
            continue

        # normalize to 100 at first valid point
        series_norm = series / series.loc[first_valid] * 100
        raw_exchange_rate = 1/df[panusatls_col]

        for date, index_val, er_val in zip(series_norm.index, series_norm.values, raw_exchange_rate.values):
            if pd.isna(index_val):
                continue
            records.append({
                "country": country,
                "year": date.year,
                "hybrid_index": float(index_val),
                "nominal_ER": float(er_val)
            })

    tidy_df = pd.DataFrame(records).sort_values(["country", "year"]).reset_index(drop=True)
    return tidy_df

# Example usage:
# hybrid_df = build_hybrid_index_df(start="1990-01-01", end="2023-01-01")
# print(hybrid_df.head())

def contiguous_runs(years):
    """Return list of slices [start_idx, end_idx] for contiguous sequences (diff==1)."""
    years = np.asarray(years)
    if len(years) == 0:
        return []
    cuts = np.where(np.diff(years) != 1)[0] + 1
    starts = np.r_[0, cuts]
    ends   = np.r_[cuts, len(years)]
    return list(zip(starts, ends))  # half-open intervals

def compute_terminal_residuals(
    df,
    years=15,
    discount_rate=0.05,
    window_mode="two_non_overlapping"  # or "rolling"
):
    """
    Compute terminal residuals (PV of net normalized deviations) for fixed-horizon windows.
    
    df must have: 'country', 'year', 'hybrid_index'

    Returns DataFrame with:
      country, window_start, window_end, baseline_value, net_pv, loss
    """
    out = []
    af = (1 - (1 + discount_rate) ** -years) / discount_rate
    
    for country, g in df.groupby("country"):
        g = g.sort_values("year").reset_index(drop=True)
        n = len(g)
        
        if n < years:
            continue

        # choose window starts
        if window_mode == "two_non_overlapping":
            starts = list(range(0, n - years + 1, years))[:2]  # up to 2 per run
        elif window_mode == "rolling":
            starts = list(range(0, n - years + 1))
        else:
            raise ValueError("window_mode must be 'two_non_overlapping' or 'rolling'")

        for i0 in starts:
            win = g.iloc[i0:i0+years].copy()

            # skip windows with missing values in hybrid_index
            if win["hybrid_index"].isna().any():
                continue

            base = float(win["hybrid_index"].iloc[0])
            base_nominal = float(win["nominal_ER"].iloc[0])

            # signed normalized deviation
            dev = (win["hybrid_index"].to_numpy() / base) - 1.0
            dev_nominal = (win["nominal_ER"].to_numpy() / base_nominal) - 1.0

            # discount factors to start of window (year-end cashflows)
            t = np.arange(1, years + 1, dtype=float)
            dfactors = (1.0 + discount_rate) ** (-t)

            net_pv_hybrid = float(np.sum(dev * dfactors))
            net_pv_nominal= float(np.sum(dev_nominal * dfactors))
            loss   = min(net_pv_hybrid, 0.0)

            out.append({
                "country": country,
                "window_start": int(win["year"].iloc[0]),
                "window_end": int(win["year"].iloc[-1]),
                "baseline_value": base,
                "net_pv_per_annual_debt_service": net_pv_hybrid,
                "net_pv_by_total_debt_stock": net_pv_hybrid/af,
                "net_pv_nominal_per_annual_debt_service": net_pv_nominal,
                "net_pv_nominal_by_total_debt_stock": net_pv_nominal/af,
                "loss": loss,    
                "loss_per_debt_stock": loss / af   
            })

    return pd.DataFrame(out)
# Example usage:
# residuals = compute_terminal_residuals(df, years=15, discount_rate=0.05,
#                                        window_mode="two_non_overlapping")
# expected_loss = residuals["loss"].mean()
# prob_positive = (residuals["loss"] > 0).mean()

def annuity_factor(n_years=15, r=0.05):
    return (1 - (1 + r) ** -n_years) / r


In [ ]:
rho = 0.1
tenor = 15
hybrid_df = build_hybrid_index_df(start="1990-01-01", end="2023-01-01")
residuals = compute_terminal_residuals(hybrid_df, years=tenor, discount_rate=rho, window_mode="rolling")
af = annuity_factor(n_years=tenor, r=rho)

prob_negative = (residuals["loss"] < 0).mean()
expected_loss = residuals["loss"].mean()
annual_premium = expected_loss / af**2
annual_premium, expected_loss/af

In [ ]:
residuals["net_pv_nominal_by_total_debt_stock"]

In [ ]:
residuals.sort_values("net_pv_by_total_debt_stock")
